# PhyGrainNet — Kaggle runner
Attach **soil-grain-size-from-photos**, enable Internet for setup, and choose a GPU for neural runs. The code uses one GPU; selecting T4 x2 does not combine their memory.

Default Run All: audit → saved A000 baselines → one-fold GPU smoke test → archive. Full CV and external pretraining are opt-in below. Smoke scores are not model-selection evidence.

For code developed locally, attach the verified source ZIP and set SOURCE_ARCHIVE to its actual mount path. GitHub main does not automatically contain workspace edits.


In [ ]:
from pathlib import Path
import os, subprocess, sys, json
BRANCH = "main"  # Must contain the reviewed code; optionally set CODE_REF to its commit.
CODE_REF = None
SOURCE_ARCHIVE = None  # Set actual /kaggle/input/.../phygrainnet_source.zip for this local release.
REPO = Path('/kaggle/working/PhyGrainNet')
DATA_ROOT = Path('/kaggle/input/soil-grain-size-from-photos')
RUN_SMOKE = True
RUN_CLASSICAL = False
RUN_FULL_CV = False
RUN_SSL = False
RUN_ETS = False
RUN_ENSEMBLE = False

def checked(args, log_name=None):
    if log_name is None:
        subprocess.run([str(a) for a in args], check=True)
    else:
        log = Path('/kaggle/working/outputs/logs') / log_name
        log.parent.mkdir(parents=True, exist_ok=True)
        with log.open('w') as f:
            proc = subprocess.Popen([str(a) for a in args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            for line in proc.stdout:
                print(line, end=''); f.write(line); f.flush()
            if proc.wait():
                raise RuntimeError(f'Command failed; see {log}')

if SOURCE_ARCHIVE:
    import zipfile, hashlib
    assert not REPO.exists(), 'Use an empty REPO path to avoid mixing source versions.'
    with zipfile.ZipFile(SOURCE_ARCHIVE) as archive:
        manifest = json.loads(archive.read('source_manifest.json'))
        assert set(archive.namelist()) == set(manifest['files']) | {'source_manifest.json'}, 'Unexpected archive members'
        base = REPO.resolve()
        for name, info in manifest['files'].items():
            target = (base / name).resolve()
            assert target.is_relative_to(base), 'Unsafe archive path'
            payload = archive.read(name)
            assert hashlib.sha256(payload).hexdigest() == info['sha256'], f'Checksum mismatch: {name}'
            assert len(payload) == info['size'], f'Size mismatch: {name}'
        archive.extractall(REPO)
elif not REPO.exists():
    checked(['git', 'clone', '-b', BRANCH, 'https://github.com/rexxar280903/PhyGrainNet.git', REPO])
if CODE_REF and not SOURCE_ARCHIVE:
    checked(['git', '-C', REPO, 'checkout', CODE_REF])
os.chdir(REPO)
checked([sys.executable, '-m', 'pip', 'install', '-e', '.', 'pytest'])
if not SOURCE_ARCHIVE:
    checked(['git', 'log', '--oneline', '-1'])
else:
    print('Source manifest SHA256:', hashlib.sha256((REPO / 'source_manifest.json').read_bytes()).hexdigest())
assert all((REPO / f).exists() for f in ['scripts/export_results.py', 'scripts/profile_model.py', 'scripts/research_report.py']), 'Use the v0.3 source bundle or branch/commit containing this release.'
assert (DATA_ROOT / 'Training_labels_updated.csv').exists(), 'Attach the competition input or correct DATA_ROOT.'
import torch
print('Python:', sys.version, 'Torch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


## 1. Dataset audit
A failed audit stops this notebook. Review size/PPM warnings and duplicate risks before full training. Exact duplicates are checked; near-duplicate review remains manual.

In [ ]:
checked([sys.executable, 'scripts/audit_dataset.py', '--data-root', DATA_ROOT], 'dataset_audit.log')

## 2. A000 — saved mean/median baselines
Uses the same five grouped folds and seed as A001/C100. Saves targets, OOF, test curves, metrics, config, groups and registry rows.

In [ ]:
checked([sys.executable, 'scripts/baseline_prior.py', '--data-root', DATA_ROOT], 'A000.log')

## Compute profile before training
Counts are partial Conv/Linear MACs. Synthetic timings include AdamW/EMA but exclude image IO, augmentation and evaluation. Import outputs/profile.json into the offline dashboard. Actual training history records loader-inclusive seconds_per_step. One GPU is used, even if T4 x2 is allocated.


In [ ]:
if RUN_SMOKE or RUN_FULL_CV:
    assert torch.cuda.is_available(), 'Enable GPU for a GPU benchmark.'
    checked([sys.executable, 'scripts/profile_model.py', '--benchmark', '--device', 'cuda',
             '--out', '/kaggle/working/outputs/profile.json'], 'profile.log')


## 3. GPU smoke test
One epoch, two steps, one fold, separate experiment ID. Check memory, loss, timing and saved checkpoint before enabling full CV.

In [ ]:
if RUN_SMOKE:
    assert torch.cuda.is_available(), 'Enable a GPU or set RUN_SMOKE=False for a CPU baseline session.'
    checked([sys.executable, 'scripts/train.py', '--config', 'configs/phygrainnet/kaggle_scratch.yaml',
             f'data.root={DATA_ROOT}', 'experiment_id=C100_smoke', 'training.epochs=1',
             'training.steps_per_epoch=2', 'training.batch_size=1', 'training.tiles_per_sample=2',
             'cv.max_folds=1', 'inference.tta=1', 'inference.max_tiles_per_image=4', 'inference.chunk=8'], 'C100_smoke.log')


## 4. Full baseline and scratch CV (opt-in)
Enable RUN_CLASSICAL and RUN_FULL_CV after reviewing the audit and smoke run. Full runs retain all folds. Checkpoints store inference weights; interrupted epochs cannot resume optimizer state.

In [ ]:
if RUN_CLASSICAL:
    checked([sys.executable, 'scripts/train.py', '--config', 'configs/classical/features_v1.yaml', f'data.root={DATA_ROOT}', 'cv.strategy=group_kfold', 'cv.n_folds=5'], 'A001.log')
if RUN_FULL_CV:
    assert torch.cuda.is_available(), 'Enable GPU for full CV.'
    checked([sys.executable, 'scripts/train.py', '--config', 'configs/phygrainnet/kaggle_scratch.yaml', f'data.root={DATA_ROOT}'], 'C100.log')
    checked([sys.executable, 'scripts/make_submission.py', '--run', '/kaggle/working/outputs/C100',
             '--data-root', DATA_ROOT, '--out', '/kaggle/working/submission_C100.csv'])


## 5. Optional SSL / ETS pretraining
Keep disabled for the first session. P100 defaults to the P000 checkpoint. ETS requires a prepared input dataset. SSL uses all unlabelled competition photos, including validation photos: report this as transductive CV, and do not claim strictly inductive validation.

In [ ]:
if RUN_SSL:
    checked([sys.executable, 'scripts/train.py', '--config', 'configs/pretrain/ssl.yaml', f'data.root={DATA_ROOT}'], 'P000.log')
    checked([sys.executable, 'scripts/train.py', '--config', 'configs/phygrainnet/mv_ssl.yaml', f'data.root={DATA_ROOT}',
             'training.batch_size=2', 'training.tiles_per_sample=6', 'inference.chunk=16'], 'D100.log')
if RUN_ETS:
    ext = Path('/kaggle/input/ets-photogranulometry-10ppm')
    assert all((ext / f).exists() for f in ['ets_catalog.csv', 'ets_labels_11pt.csv']), 'Attach prepared ETS data.'
    assert Path('/kaggle/working/outputs/P000/ssl_encoder.pt').exists(), 'Run P000 first, or override P100 init_checkpoint=null.'
    checked([sys.executable, 'scripts/train.py', '--config', 'configs/pretrain/ets.yaml', f'data.root={DATA_ROOT}',
             'training.batch_size=2', 'training.tiles_per_sample=6', 'inference.chunk=16'], 'P100.log')
    checked([sys.executable, 'scripts/train.py', '--config', 'configs/phygrainnet/mv_ets.yaml', f'data.root={DATA_ROOT}',
             'training.batch_size=2', 'training.tiles_per_sample=6', 'inference.chunk=16'], 'D200.log')


## 6. Optional ensemble
Requires complete OOF coverage from all candidate runs. One-fold smoke outputs are rejected.

In [ ]:
if RUN_ENSEMBLE:
    checked([sys.executable, 'scripts/ensemble.py', '--runs', '/kaggle/working/outputs/A001_ridge10',
             '/kaggle/working/outputs/C100', '--data-root', DATA_ROOT, '--out', '/kaggle/working/submission_H001.csv'], 'H001.log')
    checked([sys.executable, 'scripts/validate.py', '/kaggle/working/submission_H001.csv', '--data-root', DATA_ROOT])


## 7. Review saved evaluation and archive
Download phygrainnet_results.zip for a local backup. It includes checkpoints, predictions, targets, groups, metrics, resolved configs, logs, registry, audit, source snapshot and checksum manifest. Raw competition images and rebuildable cache are excluded.

For persistent Kaggle outputs use Save Version → Save & Run All with only the intended stages enabled; verify the saved version succeeds and its Output tab contains the archive. Import that version’s output into a later notebook to reuse checkpoints. Saving draft code alone is not a verified backup.

In [ ]:
import pandas as pd
registry = pd.read_csv('/kaggle/working/registry.csv')
display(registry.sort_values('cv_emd'))
if RUN_FULL_CV:
    checked([sys.executable, 'scripts/research_report.py', '--runs', '/kaggle/working/outputs/C100',
             '--out', '/kaggle/working/outputs/research_report.json'])
checked([sys.executable, 'scripts/export_results.py'])
from IPython.display import FileLink, display
display(FileLink('/kaggle/working/phygrainnet_results.zip'))


## 8. Restore inference in a later session
Attach the saved notebook output, then run (adjust its mounted folder):
```python
from glob import glob
checkpoints = sorted(glob('/kaggle/input/<saved-notebook>/outputs/C100/fold*.pt'))
assert checkpoints
checked([sys.executable, 'scripts/predict.py', '--checkpoints', *checkpoints,
         '--data-root', DATA_ROOT, '--out', '/kaggle/working/submission_C100_restored.csv'])
```
Compare the restored CSV against the original saved submission. Promote results only after complete CV and clean-session restore pass. Copy completed registry rows to experiments/registry.csv in GitHub with the matching code commit.
